# Binary Class Penguin Logistic Regression (PyTorch)
逻辑回归的pytorch版本

In [1]:
import numpy as np
import torch

In [2]:
# read the variables: class label and features

def string2float(item):
    return float(item) if item != '' else None

def read_strings(filename, col):
    with open(filename) as f:
        lines = f.readlines()
    values = [line.strip().split(',')[col] for line in lines[1:]]
    return values

def read_floats(filename, col):
    with open(filename) as f:
        lines = f.readlines()
    values = [string2float(line.strip().split(',')[col]) for line in lines[1:]]
    return values

def read_data_and_label(filename):
    x1 = np.array(read_floats(filename, col=4))  # flipper length
    x2 = np.array(read_floats(filename, col=5))  # body mass
    y = np.array(read_strings(filename, col=0))  # species

    y = [s == 'Gentoo' for s in y]
    y = np.array(y).astype(float)

    idx = (x1 != None) & (x2 != None) & (y != None)
    x1 = x1[idx]
    x2 = x2[idx]
    y = y[idx]

    X = np.stack((x1, x2), axis=1).astype(float)
    y = y.reshape((y.size, 1)).astype(float)

    return X, y

In [3]:
def min_max_normalize(X, min_val=None, max_val=None):
    X = np.array(X)
    if min_val is None:
        min_val = np.min(X, axis=0, keepdims=True)
    if max_val is None:
        max_val = np.max(X, axis=0, keepdims=True)
    range_val = max_val - min_val
    range_val = np.where(range_val == 0, 1, range_val)
    X_norm = (X - min_val) / range_val
    return X_norm, min_val, max_val

In [4]:
def z_score_normalize(X, mean=None, std=None):
    if mean is None:
        mean = np.mean(X, axis=0)
    if std is None:
        std = np.std(X, axis=0)
    std = np.where(std == 0, 1, std)
    X_norm = (X - mean) / std
    return X_norm, mean, std

In [5]:
def print_formatted_vector(vec):
    s = ' '.join(f"{x.item():.4f}" for x in vec)
    print(s)

In [14]:
class LogisticRegressionModel(torch.nn.Module):
    def __init__(self, in_dim, out_dim):
        #这里的super就是继承父类
        super().__init__()
        self.linear = torch.nn.Linear(in_dim, out_dim)

    def forward(self, x):
        x = self.linear(x)
        x = torch.sigmoid(x)
        return x

In [15]:
# training data›

filename = '/Users/liubingyi/Documents/learn/data/palmer-penguins-train.txt'
X, y = read_data_and_label(filename)

X, X_mean, X_std = z_score_normalize(X)
#从numpy的nd.array改成pytorch的数据格式
X = torch.from_numpy(X).float()
y = torch.from_numpy(y).float()

In [16]:
# init training model
#随机初始化
torch.manual_seed(0)

model = LogisticRegressionModel(in_dim=X.shape[1], out_dim=1)

print(model.linear.weight)
print(model.linear.bias)

Parameter containing:
tensor([[-0.0053,  0.3793]], requires_grad=True)
Parameter containing:
tensor([-0.5820], requires_grad=True)


In [17]:
def criterion(y_pred, y):
    epsilon = 1e-15
    y_pred = torch.clip(y_pred, epsilon, 1 - epsilon)
    losses = -(y*torch.log(y_pred) + (1 - y)*torch.log(1 - y_pred))
    return torch.mean(losses)

# criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

In [18]:
# training

for epoch in range(2000):
    optimizer.zero_grad()
    y_pred = model(X)

    loss = criterion(y_pred, y)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 200 == 0:
        print('epoch {}: loss {:.4f}'.format(epoch + 1, loss.item()))
        print_formatted_vector(y[::20])
        print_formatted_vector(y_pred[::20])

epoch 200: loss 0.0871
0.0000 0.0000 0.0000 0.0000 1.0000 1.0000 1.0000 0.0000 0.0000
0.0257 0.0057 0.0158 0.0107 0.8443 0.9840 0.9669 0.1571 0.1252
epoch 400: loss 0.0655
0.0000 0.0000 0.0000 0.0000 1.0000 1.0000 1.0000 0.0000 0.0000
0.0101 0.0016 0.0058 0.0036 0.8800 0.9929 0.9827 0.1052 0.0784
epoch 600: loss 0.0568
0.0000 0.0000 0.0000 0.0000 1.0000 1.0000 1.0000 0.0000 0.0000
0.0055 0.0007 0.0030 0.0018 0.8945 0.9955 0.9880 0.0787 0.0562
epoch 800: loss 0.0519
0.0000 0.0000 0.0000 0.0000 1.0000 1.0000 1.0000 0.0000 0.0000
0.0035 0.0004 0.0018 0.0010 0.9028 0.9968 0.9907 0.0624 0.0432
epoch 1000: loss 0.0486
0.0000 0.0000 0.0000 0.0000 1.0000 1.0000 1.0000 0.0000 0.0000
0.0024 0.0002 0.0012 0.0007 0.9086 0.9975 0.9924 0.0513 0.0347
epoch 1200: loss 0.0463
0.0000 0.0000 0.0000 0.0000 1.0000 1.0000 1.0000 0.0000 0.0000
0.0017 0.0002 0.0009 0.0005 0.9129 0.9980 0.9935 0.0434 0.0288
epoch 1400: loss 0.0445
0.0000 0.0000 0.0000 0.0000 1.0000 1.0000 1.0000 0.0000 0.0000
0.0013 0.0001 0.0

In [19]:
print(model.linear.weight)
print(model.linear.bias)

Parameter containing:
tensor([[3.0237, 3.3709]], requires_grad=True)
Parameter containing:
tensor([-2.4761], requires_grad=True)


In [20]:
# test data

filename = '/Users/liubingyi/Documents/learn/data/palmer-penguins-test.txt'

X, y = read_data_and_label(filename)
#对测试集的特征 X 进行 Z-score 标准化
X, _, _ = z_score_normalize(X, X_mean, X_std)

X = torch.from_numpy(X).float()
y = torch.from_numpy(y).float()

In [21]:
# evaluation

model.eval()

with torch.no_grad():
    y_pred = model(X)
    #这是将预测的概率值转换为二分类标签（0 或 1）。如果预测的概率大于 0.5，认为预测为 1，否则为 0
    y_pred = (y_pred > 0.5).float()
    y_corr = y == y_pred
    print(torch.mean(y_corr.float()))  # accuarcy

tensor(0.9000)
